# Exp.02 — PGR²M residual-dropout τ sensitivity: matched 300k training queue

## Goal

This experiment directly extends the manuscript's residual-dropout ablation.
The new matched runs are

\[
\tau \in \{0.1,\;0.3,\;0.7,\;0.9\}.
\]

The `τ=0.1` model is **retrained as a matched control**.  The previously
reconstructed 300k `τ=0.1` model is kept as a reproduction/reference model,
but it is not mixed into the new sensitivity comparison because the public
training code seeds NumPy and PyTorch but does not explicitly seed Python's
`random` module, while RVQ quantizer dropout uses Python `random`.

### Why this requires retraining

`pdrop_res` is a **training-time** stochastic residual mask.  Changing τ only
at inference is not a valid sensitivity test.  All four models therefore start
from scratch and use the same 300k training protocol.

### Frozen common protocol

- HumanML3D
- batch size 256
- LR `2e-4` (manuscript-faithful)
- 300,000 iterations
- warm-up 1,000
- validation every 1,000
- six RVQ stages
- residual codebook size 512
- `rvq_quantize_dropout_prob=0.2`
- seed 123 for **Python random + NumPy + PyTorch**
- same reconstructed `losses.yaml`
- one A100 GPU, runs executed **sequentially**, never concurrently

The notebook first performs a tiny `τ=0.9` production-batch smoke test, then
launches the four 300k runs as one detached queue with `nohup`.

## Important interpretation

The post-training audit in this notebook uses **validation-set metrics only**
to verify training and checkpoint selection.  Those numbers should **not** be
copied into the paper's final residual-dropout table.  After the four runs
finish, the next step will be Exp.02B: the paper-style HumanML3D **test-set**
evaluation of the selected best-FID checkpoints, including repeated evaluation
for mean/95% CI and orthogonality.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json, hashlib, shutil, os, shlex, time, re, sys

PGR = Path(".")
PRE = PGR / "revision_preflight"
LOGDIR = PGR / "revision_logs" / "exp02_tau"
OUT_ROOT = PGR / "exp_result_revision_tau_sensitivity"
RESULTS = PGR / "revision_results" / "exp02_tau"

PY = Path("venv/bin/python")
BASE_WRAPPER_CANDIDATES = [
    PRE / "train_pg_tokenizer_revision_compat.py",
    PRE / "train_pg_tokenizer_compat.py",
]
SEEDED_WRAPPER = PRE / "train_pg_tokenizer_exp02_seeded_compat.py"

LOSS_CFG = PGR / "configs/exp_dec/losses.yaml"

QUEUE_CONFIG = PRE / "exp02_tau_queue_config.json"
QUEUE_RUNNER = PRE / "exp02_tau_queue_runner.py"
QUEUE_STATE = LOGDIR / "exp02_tau_queue_state.json"
QUEUE_MASTER_LOG = LOGDIR / "exp02_tau_queue_master.log"
QUEUE_PIDFILE = LOGDIR / "exp02_tau_queue.pid"
FREEZE = RESULTS / "exp02_tau_config_freeze.json"

TAUS = [0.1, 0.3, 0.7, 0.9]
SEED = 123

for d in [PRE, LOGDIR, OUT_ROOT, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

print("Python   :", PY)
print("PGR root :", PGR)
print("Output   :", OUT_ROOT)
print("Taus     :", TAUS)


## 1. Hard gate: completed reproduction assets


In [ ]:
base_wrapper = next((p for p in BASE_WRAPPER_CANDIDATES if p.exists()), None)

required = [
    PY,
    PGR / "train_pg_tokenizer.py",
    PGR / "dataset/dataset_PC.py",
    PGR / "dataset/dataset_TM_eval.py",
    LOSS_CFG,
]
missing = [str(p) for p in required if not p.exists()]

if base_wrapper is None:
    missing.append("one of: " + ", ".join(str(p) for p in BASE_WRAPPER_CANDIDATES))

if missing:
    print("[FAILED] Missing required assets:")
    for p in missing:
        print(" ", p)
    raise RuntimeError("Exp.02 cannot start until the reproduction assets are present.")

print("[OK] base compatibility wrapper:", base_wrapper)

# Confirm the completed Exp00T run is present, but do not reuse its weights.
exp00t = PGR / "revision_logs" / "exp00t_latest_launch.json"
if exp00t.exists():
    info = json.loads(exp00t.read_text())
    print("[OK] Exp00T launch record:", exp00t)
    print("     prior output root:", info.get("output_root"))
else:
    print("[WARN] Exp00T launch record not found; Exp.02 can still proceed if assets are intact.")


## 2. Create a deterministic compatibility wrapper

The already validated S12/Exp00T compatibility launcher is preserved.  Exp.02
adds exactly one experiment-control change: `random.seed(123)` is injected
immediately before the compatibility launcher enters the public
`train_pg_tokenizer.py`.

This makes the RVQ quantizer-dropout RNG matched across τ conditions while
leaving PGR²M source files untouched.


In [ ]:
src = base_wrapper.read_text()

# Insert the Python-random seed immediately before the LAST runpy.run_path(...)
# call, which is the hand-off into public train_pg_tokenizer.py in the validated
# compatibility launcher.
lines = src.splitlines()
runpy_indices = [i for i, line in enumerate(lines) if "runpy.run_path" in line]

if not runpy_indices:
    raise RuntimeError("Validated compatibility wrapper has no runpy.run_path call.")

insert_at = runpy_indices[-1]
seed_block = [
    "",
    "# Exp.02 matched-RNG control: seed Python random immediately before training.",
    "import random as _exp02_python_random",
    f"_exp02_python_random.seed({SEED})",
    "",
]

# Avoid injecting the block twice if this notebook is rerun.
if "_exp02_python_random.seed" not in src:
    lines[insert_at:insert_at] = seed_block

SEEDED_WRAPPER.write_text("\n".join(lines) + "\n")
compile(SEEDED_WRAPPER.read_text(), str(SEEDED_WRAPPER), "exec")

print("[OK] seeded wrapper:", SEEDED_WRAPPER)
print("Contains random seed:", f"_exp02_python_random.seed({SEED})" in SEEDED_WRAPPER.read_text())
print("Public source modified: False")


## 3. Freeze the exact four 300k commands


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def tau_tag(tau):
    return str(tau).replace(".", "p")

COMMON_ARGS = [
    "--batch-size", "256",
    "--lr", "2e-4",
    "--total-iter", "300000",
    "--warm-up-iter", "1000",
    "--eval-iter", "1000",
    "--print-iter", "200",
    "--lr-scheduler", "200000",
    "--nb-code", "392",
    "--down-t", "2",
    "--depth", "3",
    "--dilation-growth-rate", "3",
    "--dataname", "t2m",
    "--vq-act", "relu",
    "--loss-cfg-path", "./configs/exp_dec/losses.yaml",
    "--use-keywords",
    "--output-emb-width", "392",
    "--rvq-num-quantizers", "6",
    "--rvq-quantize-dropout-prob", "0.2",
    "--rvq-quantize-dropout-cutoff-index", "0",
    "--rvq-nb-code", "512",
    "--rvq-quantizer-type", "soft",
    "--rvq-vq-loss-beta", "0.25",
    "--rvq-loss-weight", "0.02",
    "--detach-p-latent",
    "--params-soft-ent-loss", "0.01",
    "--unuse-ema",
    "--num_workers", "4",
    "--seed", str(SEED),
]

runs = []
for tau in TAUS:
    exp_name = f"exp_pg_tokenizer_tau_{tau_tag(tau)}_seed{SEED}"
    cmd = [
        str(PY), "-u", str(SEEDED_WRAPPER),
        "--out-dir", str(OUT_ROOT),
        "--exp-name", exp_name,
        "--pdrop-res", str(tau),
        *COMMON_ARGS,
    ]
    runs.append({
        "tau": tau,
        "tag": tau_tag(tau),
        "exp_name": exp_name,
        "command": cmd,
    })

git_sha = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=PGR, capture_output=True, text=True
).stdout.strip()

env_probe = subprocess.run(
    [
        str(PY), "-c",
        "import sys,torch,numpy,sklearn,matplotlib;"
        "print('python='+sys.version.split()[0]);"
        "print('torch='+torch.__version__);"
        "print('cuda='+str(torch.version.cuda));"
        "print('numpy='+numpy.__version__);"
        "print('sklearn='+sklearn.__version__);"
        "print('matplotlib='+matplotlib.__version__);"
        "print('gpu='+torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"
    ],
    cwd=PGR, capture_output=True, text=True
)

freeze = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "experiment": "Exp.02 PGR2M residual-dropout tau sensitivity",
    "taus": TAUS,
    "matched_control_tau": 0.1,
    "seed_python_random": SEED,
    "seed_numpy": SEED,
    "seed_torch": SEED,
    "git_sha": git_sha,
    "environment": env_probe.stdout.strip().splitlines(),
    "python": str(PY),
    "base_compatibility_wrapper": str(base_wrapper),
    "seeded_compatibility_wrapper": str(SEEDED_WRAPPER),
    "seeded_wrapper_sha256": sha256(SEEDED_WRAPPER),
    "loss_cfg": str(LOSS_CFG),
    "loss_cfg_sha256": sha256(LOSS_CFG),
    "loss_cfg_provenance": (
        "evidence-constrained reconstruction; not proven to be the private "
        "author-original losses.yaml"
    ),
    "lr": "2e-4 manuscript-faithful",
    "runs": runs,
    "paper_metric_policy": (
        "validation metrics in this notebook are for completion/model-selection "
        "audit only; paper-facing tau table requires subsequent test-set evaluation"
    ),
}
FREEZE.write_text(json.dumps(freeze, indent=2))

print(env_probe.stdout)
print("Git SHA :", git_sha)
print("Freeze  :", FREEZE)

for r in runs:
    print(f"\nτ={r['tau']}")
    print(shlex.join(r["command"]))


## 4. Production-batch smoke — τ=0.9

This is the only synchronous gate.  It uses batch size 256 and the new seeded
wrapper.  A high τ is chosen because it strongly exercises the residual-drop
branch.  The public validation path still evaluates both pose-only and full
residual reconstruction.


In [ ]:
SMOKE_LOG = LOGDIR / f"exp02_smoke_tau0p9_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
SMOKE_OUT = PRE / "exp02_tau_smoke"

SMOKE_CMD = [
    str(PY), "-u", str(SEEDED_WRAPPER),
    "--batch-size", "256",
    "--lr", "2e-4",
    "--total-iter", "1",
    "--warm-up-iter", "2",
    "--eval-iter", "1",
    "--print-iter", "200",
    "--lr-scheduler", "200000",
    "--nb-code", "392",
    "--down-t", "2",
    "--depth", "3",
    "--dilation-growth-rate", "3",
    "--dataname", "t2m",
    "--vq-act", "relu",
    "--loss-cfg-path", "./configs/exp_dec/losses.yaml",
    "--out-dir", str(SMOKE_OUT),
    "--exp-name", "exp02_tau0p9_seeded_smoke",
    "--use-keywords",
    "--output-emb-width", "392",
    "--rvq-num-quantizers", "6",
    "--rvq-quantize-dropout-prob", "0.2",
    "--rvq-quantize-dropout-cutoff-index", "0",
    "--rvq-nb-code", "512",
    "--rvq-quantizer-type", "soft",
    "--rvq-vq-loss-beta", "0.25",
    "--rvq-loss-weight", "0.02",
    "--detach-p-latent",
    "--params-soft-ent-loss", "0.01",
    "--pdrop-res", "0.9",
    "--unuse-ema",
    "--num_workers", "4",
    "--seed", str(SEED),
]

print("[RUN] τ=0.9 seeded batch-256 smoke")
print(shlex.join(SMOKE_CMD))

with open(SMOKE_LOG, "w") as f:
    smoke = subprocess.run(
        SMOKE_CMD,
        cwd=PGR,
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=7200,
    )

print("returncode:", smoke.returncode)
print("log:", SMOKE_LOG)
print("\n=== SMOKE LOG TAIL ===")
print("\n".join(SMOKE_LOG.read_text(errors="replace").splitlines()[-120:]))

SMOKE_OK = (smoke.returncode == 0)
print("\nSMOKE_OK:", SMOKE_OK)

if not SMOKE_OK:
    raise RuntimeError("Exp.02 smoke failed. Do not launch the 300k queue.")


## 5. Write the resumable sequential queue runner

The queue executes **one τ at a time**.  If a run fails, later τ values are not
started.  The queue state is saved after every transition.

Re-running this notebook after a completed τ does not automatically retrain it:
a condition marked `complete` in the queue state is skipped unless the state
file is manually removed.


In [ ]:
config = {
    "cwd": str(PGR),
    "python": str(PY),
    "out_root": str(OUT_ROOT),
    "logdir": str(LOGDIR),
    "state_path": str(QUEUE_STATE),
    "total_iter": 300000,
    "runs": runs,
}
QUEUE_CONFIG.write_text(json.dumps(config, indent=2))

runner_source = r'''
from pathlib import Path
from datetime import datetime
import subprocess, json, re, sys, traceback

CFG_PATH = Path(sys.argv[1])
cfg = json.loads(CFG_PATH.read_text())

cwd = Path(cfg["cwd"])
out_root = Path(cfg["out_root"])
logdir = Path(cfg["logdir"])
state_path = Path(cfg["state_path"])
total_iter = int(cfg["total_iter"])

logdir.mkdir(parents=True, exist_ok=True)
out_root.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now().isoformat(timespec="seconds")

def load_state():
    if state_path.exists():
        try:
            return json.loads(state_path.read_text())
        except Exception:
            pass
    return {
        "created_at": now(),
        "queue_status": "pending",
        "runs": {}
    }

def save_state(state):
    tmp = state_path.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2))
    tmp.replace(state_path)

def max_train_iter(log_path):
    if not log_path.exists():
        return None
    text = log_path.read_text(errors="replace")
    vals = [
        int(x) for x in re.findall(
            r"^\d{4}-\d{2}-\d{2} .*? Train\. Iter\s+(\d+)\s*:",
            text,
            flags=re.M
        )
    ]
    return max(vals) if vals else None

state = load_state()
state["queue_status"] = "running"
state["started_or_resumed_at"] = now()
save_state(state)

for spec in cfg["runs"]:
    tau = float(spec["tau"])
    key = str(tau)
    exp_name = spec["exp_name"]
    cmd = spec["command"]

    prior = state["runs"].get(key, {})
    if prior.get("status") == "complete":
        print(f"[SKIP] tau={tau} already complete", flush=True)
        continue

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_log = logdir / f"exp02_tau_{spec['tag']}_{stamp}.log"

    state["runs"][key] = {
        "tau": tau,
        "exp_name": exp_name,
        "status": "running",
        "started_at": now(),
        "log": str(run_log),
        "command": cmd,
    }
    save_state(state)

    print(f"[START] tau={tau}", flush=True)
    print(" ".join(cmd), flush=True)

    try:
        with open(run_log, "w") as f:
            proc = subprocess.run(
                cmd,
                cwd=cwd,
                stdout=f,
                stderr=subprocess.STDOUT,
                text=True,
            )
        rc = int(proc.returncode)
    except Exception as e:
        rc = -999
        with open(run_log, "a") as f:
            f.write("\nQUEUE RUNNER EXCEPTION:\n")
            f.write(traceback.format_exc())

    max_iter = max_train_iter(run_log)

    exp_root = out_root / exp_name
    dirs = sorted(
        [p for p in exp_root.glob("*") if p.is_dir()],
        key=lambda p: p.stat().st_mtime
    ) if exp_root.exists() else []
    run_dir = dirs[-1] if dirs else None

    checkpoint_exists = bool(run_dir and (run_dir / "net_last.pth").exists())
    complete = (rc == 0 and max_iter == total_iter and checkpoint_exists)

    rec = state["runs"][key]
    rec.update({
        "returncode": rc,
        "max_train_iter": max_iter,
        "run_dir": str(run_dir) if run_dir else None,
        "net_last_exists": checkpoint_exists,
        "finished_at": now(),
        "status": "complete" if complete else "failed",
    })
    save_state(state)

    print(
        f"[END] tau={tau} rc={rc} max_iter={max_iter} "
        f"checkpoint={checkpoint_exists} status={rec['status']}",
        flush=True
    )

    if not complete:
        state["queue_status"] = "failed"
        state["failed_tau"] = tau
        state["finished_at"] = now()
        save_state(state)
        sys.exit(1)

state["queue_status"] = "complete"
state["finished_at"] = now()
save_state(state)
print("[QUEUE COMPLETE] all tau runs finished", flush=True)
'''

QUEUE_RUNNER.write_text(runner_source)
compile(QUEUE_RUNNER.read_text(), str(QUEUE_RUNNER), "exec")

print("Queue config:", QUEUE_CONFIG)
print("Queue runner:", QUEUE_RUNNER)
print("State       :", QUEUE_STATE)


## 6. Launch the detached 4-run queue

**Running this cell starts the actual Exp.02 training queue.**

The browser and Jupyter notebook may be closed afterwards.  A PID guard
prevents launching a second queue while the recorded queue process is alive.


In [ ]:
if not SMOKE_OK:
    raise RuntimeError("Smoke gate is not green.")

if QUEUE_PIDFILE.exists():
    try:
        old_pid = int(QUEUE_PIDFILE.read_text().strip())
    except Exception:
        old_pid = None

    if old_pid is not None:
        alive = subprocess.run(
            ["bash", "-lc", f"kill -0 {old_pid} 2>/dev/null"],
            capture_output=True
        ).returncode == 0

        if alive:
            raise RuntimeError(
                f"Exp.02 queue is already alive: PID={old_pid}. "
                "A duplicate queue will not be launched."
            )

launch_cmd = (
    f"nohup {shlex.quote(str(PY))} -u {shlex.quote(str(QUEUE_RUNNER))} "
    f"{shlex.quote(str(QUEUE_CONFIG))} "
    f"> {shlex.quote(str(QUEUE_MASTER_LOG))} 2>&1 < /dev/null & echo $!"
)

p = subprocess.run(
    ["bash", "-lc", launch_cmd],
    cwd=PGR,
    capture_output=True,
    text=True,
    timeout=30,
)

if p.returncode != 0:
    print(p.stdout)
    print(p.stderr)
    raise RuntimeError("Could not launch Exp.02 queue.")

pid = int(p.stdout.strip().splitlines()[-1])
QUEUE_PIDFILE.write_text(str(pid))

time.sleep(3)
alive = subprocess.run(
    ["bash", "-lc", f"kill -0 {pid} 2>/dev/null"],
    capture_output=True
).returncode == 0

print("=== EXP.02 QUEUE LAUNCH ===")
print("PID        :", pid)
print("Alive      :", alive)
print("Master log :", QUEUE_MASTER_LOG)
print("State      :", QUEUE_STATE)
print("PID file   :", QUEUE_PIDFILE)
print("Output root:", OUT_ROOT)

if not alive:
    if QUEUE_MASTER_LOG.exists():
        print("\n=== EARLY MASTER LOG TAIL ===")
        print("\n".join(QUEUE_MASTER_LOG.read_text(errors="replace").splitlines()[-120:]))
    raise RuntimeError("Exp.02 queue exited immediately.")


## 7. Queue status / progress

This cell is safe to re-run at any time.


In [ ]:
def extract_progress(log_path):
    log_path = Path(log_path)
    if not log_path.exists():
        return None
    txt = log_path.read_text(errors="replace")
    vals = [
        int(x) for x in re.findall(
            r"^\d{4}-\d{2}-\d{2} .*? Train\. Iter\s+(\d+)\s*:",
            txt,
            flags=re.M
        )
    ]
    return max(vals) if vals else None

pid = None
if QUEUE_PIDFILE.exists():
    try:
        pid = int(QUEUE_PIDFILE.read_text().strip())
    except Exception:
        pass

alive = False
if pid is not None:
    alive = subprocess.run(
        ["bash", "-lc", f"kill -0 {pid} 2>/dev/null"],
        capture_output=True
    ).returncode == 0

print("=== EXP.02 QUEUE STATUS ===")
print("PID   :", pid)
print("Alive :", alive)

if QUEUE_STATE.exists():
    state = json.loads(QUEUE_STATE.read_text())
    print("Queue status:", state.get("queue_status"))
    for tau in TAUS:
        rec = state.get("runs", {}).get(str(tau), {})
        if not rec:
            print(f"τ={tau}: pending")
            continue
        prog = extract_progress(rec.get("log", ""))
        pct = (100.0 * prog / 300000.0) if prog is not None else 0.0
        print(
            f"τ={tau}: status={rec.get('status')} "
            f"iter={prog} ({pct:.2f}%) "
            f"log={rec.get('log')}"
        )
else:
    print("Queue state has not been created yet.")

if QUEUE_MASTER_LOG.exists():
    print("\n=== MASTER LOG TAIL ===")
    print("\n".join(QUEUE_MASTER_LOG.read_text(errors="replace").splitlines()[-80:]))


### Useful shell commands

```bash
cat revision_logs/exp02_tau/exp02_tau_queue_state.json
tail -f revision_logs/exp02_tau/exp02_tau_queue_master.log
ps -fp "$(cat revision_logs/exp02_tau/exp02_tau_queue.pid)"
nvidia-smi
```

To inspect the currently running τ-specific log, first read the queue-state
JSON and then `tail -f` the `log` path of the condition whose status is
`running`.


## 8. Post-run validation audit

Run this section **only after** the queue state reports `complete`.

This audit verifies all four 300k runs, parses the full-residual validation
trajectory, identifies the last FID-improvement iteration, and computes pose
codebook orthogonality from each saved best-FID checkpoint.

Again: these are **validation/model-selection diagnostics**, not the final
paper table.


In [ ]:
import numpy as np
import pandas as pd
import torch

if not QUEUE_STATE.exists():
    raise RuntimeError("No Exp.02 queue state exists.")

state = json.loads(QUEUE_STATE.read_text())
if state.get("queue_status") != "complete":
    raise RuntimeError(
        f"Queue is not complete yet: {state.get('queue_status')}. "
        "Re-run the status cell later."
    )

eval_pattern = re.compile(
    r"^\d{4}-\d{2}-\d{2} [0-9:,]+ INFO -->\s+Eva\. Iter\s+(?P<iter>\d+)\s*:,\s*"
    r"FID\.\s*(?P<fid>[-+0-9.eE]+),\s*"
    r"Diversity Real\.\s*(?P<div_real>[-+0-9.eE]+),\s*"
    r"Diversity\.\s*(?P<div>[-+0-9.eE]+),\s*"
    r"R_precision_real\.\s*\[(?P<rreal>[^\]]+)\],\s*"
    r"R_precision\.\s*\[(?P<rpred>[^\]]+)\],\s*"
    r"matching_score_real\.\s*(?P<ms_real>[-+0-9.eE]+),\s*"
    r"matching_score_pred\.\s*(?P<ms_pred>[-+0-9.eE]+),\s*"
    r"MPJPE:(?P<mpjpe>[-+0-9.eE]+),\s*"
    r"PA-MPJPE:(?P<pampjpe>[-+0-9.eE]+),\s*"
    r"ACCEL:(?P<accel>[-+0-9.eE]+)",
    flags=re.M
)

def arr3(s):
    vals = [float(v) for v in s.replace(",", " ").split()]
    return vals[:3]

def parse_full_validation(log_path):
    txt = Path(log_path).read_text(errors="replace")
    mode = "Initial"
    rows = []
    last_full_eval_iter = None
    best_fid_improvement_iter = None
    best_fid_improvement_value = None

    for line in txt.splitlines():
        if not re.match(r"^\d{4}-\d{2}-\d{2} ", line):
            continue

        if "No RVQ Validation. Iter " in line:
            mode = "PoseOnly"
        elif "Validation. Iter " in line and "No RVQ" not in line:
            mode = "Full"

        m = eval_pattern.match(line)
        if m:
            rr = arr3(m.group("rreal"))
            rp = arr3(m.group("rpred"))
            row = {
                "iter": int(m.group("iter")),
                "mode": mode,
                "fid": float(m.group("fid")),
                "diversity_real": float(m.group("div_real")),
                "diversity": float(m.group("div")),
                "top1": rp[0],
                "top2": rp[1],
                "top3": rp[2],
                "matching_score": float(m.group("ms_pred")),
                "mpjpe": float(m.group("mpjpe")),
                "pampjpe": float(m.group("pampjpe")),
                "accel": float(m.group("accel")),
            }
            rows.append(row)
            if mode == "Full":
                last_full_eval_iter = row["iter"]

        if mode == "Full" and "FID Improved from" in line:
            mm = re.search(r"FID Improved from\s+[-+0-9.eE]+\s+to\s+([-+0-9.eE]+)", line)
            if mm and last_full_eval_iter is not None:
                best_fid_improvement_iter = last_full_eval_iter
                best_fid_improvement_value = float(mm.group(1))

    frame = pd.DataFrame(rows)
    full = frame[frame["mode"] == "Full"].copy().sort_values("iter")
    return full, best_fid_improvement_iter, best_fid_improvement_value

def pose_codebook_orthogonality(ckpt_path):
    obj = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = obj.get("net", obj)

    candidates = [
        k for k in state_dict.keys()
        if k.endswith("pc_encoder.codebook.weight")
    ]
    if not candidates:
        candidates = [
            k for k in state_dict.keys()
            if "pc_encoder" in k and k.endswith("codebook.weight")
        ]
    if len(candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely identify pose codebook in {ckpt_path}: {candidates}"
        )

    w = state_dict[candidates[0]].float()
    w = w / (w.norm(dim=1, keepdim=True) + 1e-12)
    gram = w @ w.T
    n = gram.shape[0]
    mask = ~torch.eye(n, dtype=torch.bool)
    orth = gram[mask].square().mean().item()

    return orth, tuple(state_dict[candidates[0]].shape), obj.get("nb_iter")

audit_rows = []
trajectory_frames = []

for tau in TAUS:
    rec = state["runs"][str(tau)]
    if rec.get("status") != "complete":
        raise RuntimeError(f"τ={tau} is not complete.")

    run_dir = Path(rec["run_dir"])
    log_path = Path(rec["log"])
    best_ckpt = run_dir / "net_best_fid.pth"
    last_ckpt = run_dir / "net_last.pth"

    if not best_ckpt.exists() or not last_ckpt.exists():
        raise RuntimeError(f"Missing checkpoint(s) for τ={tau}: {run_dir}")

    full, best_iter, best_fid_logged5 = parse_full_validation(log_path)
    if full.empty or int(full["iter"].max()) != 300000:
        raise RuntimeError(f"τ={tau}: full validation did not reach 300000.")

    full["tau"] = tau
    trajectory_frames.append(full)

    final = full.iloc[-1]
    min_row = full.loc[full["fid"].idxmin()]
    orth, code_shape, ckpt_nb_iter = pose_codebook_orthogonality(best_ckpt)

    audit_rows.append({
        "tau": tau,
        "run_dir": str(run_dir),
        "best_fid_checkpoint": str(best_ckpt),
        "best_fid_improvement_iter": best_iter,
        "best_fid_logged_5dp": best_fid_logged5,
        "rounded_min_fid": float(min_row["fid"]),
        "rounded_min_fid_first_iter": int(min_row["iter"]),
        "checkpoint_nb_iter_if_present": ckpt_nb_iter,
        "pose_codebook_shape": str(code_shape),
        "orthogonality": orth,
        "final_iter": int(final["iter"]),
        "final_fid": float(final["fid"]),
        "final_top1": float(final["top1"]),
        "final_top2": float(final["top2"]),
        "final_top3": float(final["top3"]),
        "final_matching_score": float(final["matching_score"]),
        "final_mpjpe": float(final["mpjpe"]),
        "final_pampjpe": float(final["pampjpe"]),
        "final_accel": float(final["accel"]),
    })

audit = pd.DataFrame(audit_rows).sort_values("tau")
trajectory = pd.concat(trajectory_frames, ignore_index=True)

AUDIT_CSV = RESULTS / "exp02_validation_audit.csv"
TRAJ_CSV = RESULTS / "exp02_validation_trajectories.csv"

audit.to_csv(AUDIT_CSV, index=False)
trajectory.to_csv(TRAJ_CSV, index=False)

print("=== EXP.02 VALIDATION AUDIT ===")
print(
    audit[[
        "tau",
        "best_fid_improvement_iter",
        "best_fid_logged_5dp",
        "orthogonality",
        "final_fid",
        "final_top1",
        "final_matching_score",
        "final_mpjpe",
        "final_pampjpe",
    ]].to_string(index=False)
)

print("\nSaved:")
print(" ", AUDIT_CSV)
print(" ", TRAJ_CSV)


## 9. Final queue decision

When this is green, do **not** modify or retrain the four tokenizers.  Freeze
their best-FID checkpoints and move to **Exp.02B test-set evaluation**.


In [ ]:
expected_taus = set(TAUS)
actual_taus = set(audit["tau"].astype(float).tolist()) if "audit" in globals() else set()

FINALIZED = (
    state.get("queue_status") == "complete"
    and actual_taus == expected_taus
    and (audit["final_iter"] == 300000).all()
    and audit["best_fid_checkpoint"].map(lambda p: Path(p).exists()).all()
)

print("=== EXP.02 FINAL STATUS ===")
print("Queue complete             :", state.get("queue_status") == "complete")
print("Tau conditions             :", sorted(actual_taus))
print("All reached 300k           :", bool((audit["final_iter"] == 300000).all()))
print("Best-FID checkpoints ready :", bool(audit["best_fid_checkpoint"].map(lambda p: Path(p).exists()).all()))
print("Validation audit           :", AUDIT_CSV)
print("FINALIZED                  :", FINALIZED)

if FINALIZED:
    print("\nDECISION: EXP.02 TRAINING IS COMPLETE.")
    print("Next: Exp.02B — paper-style HumanML3D test evaluation of all four")
    print("best-FID checkpoints (mean/95% CI, Top-1, FID, MM-Dist, Orthogonality).")
else:
    raise RuntimeError("Exp.02 is not yet complete.")
